In [1]:
import os
import ssl
import certifi
import urllib3

# Configure SSL certificates
cert_path = certifi.where()
os.environ['SSL_CERT_FILE'] = cert_path
os.environ['REQUESTS_CA_BUNDLE'] = cert_path
os.environ['AWS_CA_BUNDLE'] = cert_path
os.environ['CURL_CA_BUNDLE'] = cert_path

# Create SSL context with proper certificates
ssl_context = ssl.create_default_context(cafile=cert_path)
ssl._create_default_https_context = lambda: ssl_context

# Disable SSL warnings (optional)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print(f"✅ SSL certificates configured using: {cert_path}")
print("✅ Environment variables set for AWS, requests, and curl")
print("✅ Ready to make secure HTTPS connections!")

✅ SSL certificates configured using: /Users/manojskr/Documents/Code/GitHub/graphrag-toolkit/.venv/lib/python3.10/site-packages/certifi/cacert.pem
✅ Environment variables set for AWS, requests, and curl
✅ Ready to make secure HTTPS connections!


In [2]:
%reload_ext dotenv
%dotenv

import os
from pathlib import Path

from graphrag_toolkit.lexical_graph import LexicalGraphIndex, set_logging_config
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.indexing.load import FileBasedDocs
from graphrag_toolkit.lexical_graph.indexing.build import Checkpoint

# from llama_index.readers.web import SimpleWebPageReader
from llama_index.readers.file import PDFReader

set_logging_config('INFO')

## Extract

In [3]:
extracted_docs = FileBasedDocs(
    docs_directory='extracted'
)

checkpoint = Checkpoint('extraction-checkpoint')

graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

graph_index = LexicalGraphIndex(
    graph_store, 
    vector_store
)

# doc_urls = [
#     'https://docs.aws.amazon.com/neptune/latest/userguide/intro.html',
#     'https://docs.aws.amazon.com/neptune-analytics/latest/userguide/what-is-neptune-analytics.html',
#     'https://docs.aws.amazon.com/neptune-analytics/latest/userguide/neptune-analytics-features.html',
#     'https://docs.aws.amazon.com/neptune-analytics/latest/userguide/neptune-analytics-vs-neptune-database.html'
# ]

# docs = SimpleWebPageReader(
#     html_to_text=True,
#     metadata_fn=lambda url:{'url': url}
# ).load_data(doc_urls)

# graph_index.extract(docs, handler=extracted_docs, checkpoint=checkpoint, show_progress=True)

# ---------------------------------------------------------------------------
#  PDF ingest and extraction
# ---------------------------------------------------------------------------
pdf_dir = Path("data/pdfs")
pdf_dir.mkdir(parents=True, exist_ok=True)

pdf_files = list(pdf_dir.glob("*.pdf"))
if not pdf_files:
    print(f"No PDF files found in {pdf_dir}. "
          "Add PDFs to data/pdfs/ and re-run this cell.")
else:
    print(f"Found {len(pdf_files)} PDF file(s):")
    for pdf in pdf_files:
        print(f"  • {pdf.name}")

    print("\nLoading and extracting documents …")
    # Process each PDF file individually and combine the results
    docs = []
    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file.name}:")
        file_docs = PDFReader().load_data(pdf_file)
        print(f"- Extracted {len(file_docs)} pages")
        
        # Add source metadata to each page
        for i, doc in enumerate(file_docs):
            doc.metadata.update({
                "source": str(pdf_file),
                "page_number": i + 1,
                "total_pages": len(file_docs)
            })
            docs.append(doc)
        
        print(f"- Added metadata to {len(file_docs)} pages")

    print(f"\nTotal documents to process: {len(docs)}")
    
    graph_index.extract(
        docs,
        handler=extracted_docs,
        checkpoint=checkpoint,
        show_progress=True,
    )




collection_id = extracted_docs.collection_id

print('Extraction complete')
print(f'collection_id: {collection_id}')

Found 1 PDF file(s):
  • QEM-CCR-2410-00001-VR(FQE).pdf

Loading and extracting documents …

Processing QEM-CCR-2410-00001-VR(FQE).pdf:
- Extracted 11 pages
- Added metadata to 11 pages

Total documents to process: 11
2025-07-07 12:37:37:INFO:g.l.i.e.extraction_pipeline:Running extraction pipeline [batch_size: 4, num_workers: 2]


Extracting propositions [nodes: 3, num_workers: 4]: 100%|██████████| 3/3 [00:02<00:00,  1.22it/s]
Extracting propositions [nodes: 5, num_workers: 4]: 100%|██████████| 5/5 [00:04<00:00,  1.05it/s]
Extracting topics [nodes: 5, num_workers: 4]: 100%|██████████| 5/5 [00:10<00:00,  2.14s/it]


2025-07-07 12:37:57:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch_size: 4, num_workers: 1, job_sizes: [182], batch_writes_enabled: True, batch_write_size: 25]
2025-07-07 12:37:59:INFO:g.l.i.e.extraction_pipeline:Running extraction pipeline [batch_size: 4, num_workers: 2]


Extracting propositions [nodes: 3, num_workers: 4]: 100%|██████████| 3/3 [00:02<00:00,  1.00it/s]t]
Extracting propositions [nodes: 12, num_workers: 4]: 100%|██████████| 12/12 [00:08<00:00,  1.38it/s]
Extracting topics [nodes: 12, num_workers: 4]: 100%|██████████| 12/12 [00:31<00:00,  2.66s/it]


2025-07-07 12:38:43:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch_size: 4, num_workers: 1, job_sizes: [475], batch_writes_enabled: True, batch_write_size: 25]
2025-07-07 12:38:46:INFO:g.l.i.e.extraction_pipeline:Running extraction pipeline [batch_size: 4, num_workers: 2]


Extracting propositions [nodes: 7, num_workers: 4]: 100%|██████████| 7/7 [00:05<00:00,  1.20it/s]
Extracting propositions [nodes: 6, num_workers: 4]: 100%|██████████| 6/6 [00:06<00:00,  1.02s/it]
Extracting topics [nodes: 7, num_workers: 4]: 100%|██████████| 7/7 [00:18<00:00,  2.58s/it]
Extracting propositions [nodes: 1, num_workers: 4]: 100%|██████████| 1/1 [00:02<00:00,  2.28s/it]
Extracting topics [nodes: 1, num_workers: 4]: 100%|██████████| 1/1 [00:06<00:00,  6.57s/it]


2025-07-07 12:39:22:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch_size: 4, num_workers: 1, job_sizes: [471], batch_writes_enabled: True, batch_write_size: 25]
Extraction complete
collection_id: 20250707-123731


## Build

In [4]:
%reload_ext dotenv
%dotenv

import os

from graphrag_toolkit.lexical_graph import LexicalGraphIndex, set_logging_config
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph.indexing.load import FileBasedDocs
from graphrag_toolkit.lexical_graph.indexing.build import Checkpoint

set_logging_config('INFO')

docs = FileBasedDocs(
    docs_directory='extracted',
    collection_id=collection_id
)
checkpoint = Checkpoint('build-checkpoint')

graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

graph_index = LexicalGraphIndex(
    graph_store, 
    vector_store
)

graph_index.build(docs, checkpoint=checkpoint, show_progress=True)

print('Build complete')

2025-07-07 12:57:11:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch_size: 4, num_workers: 2, job_sizes: [253, 198], batch_writes_enabled: True, batch_write_size: 25]


Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 198/198 [00:00<00:00, 43697.56it/s]
Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 253/253 [00:00<00:00, 40944.51it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 198/198 [00:00<00:00, 669195.96it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 253/253 [00:00<00:00, 808194.14it/s]


2025-07-07 12:58:05:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch_size: 4, num_workers: 2, job_sizes: [296, 107], batch_writes_enabled: True, batch_write_size: 25]


Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 107/107 [00:00<00:00, 59379.54it/s]
Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 296/296 [00:00<00:00, 68554.06it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 107/107 [00:00<00:00, 164633.36it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 296/296 [00:00<00:00, 291640.59it/s]


2025-07-07 12:58:47:INFO:g.l.i.b.build_pipeline:Running build pipeline [batch_size: 4, num_workers: 2, job_sizes: [240, 34], batch_writes_enabled: True, batch_write_size: 25]


Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 34/34 [00:00<00:00, 68429.14it/s]
Building graph [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 240/240 [00:00<00:00, 51095.53it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 34/34 [00:00<00:00, 159514.92it/s]
Building vector index [batch_writes_enabled: True, batch_write_size: 25]: 100%|██████████| 240/240 [00:00<00:00, 363667.98it/s]


Build complete
